## XGBoost

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

RuntimeError: module was compiled against NumPy C-API version 0x10 (NumPy 1.23) but the running NumPy has C-API version 0xf. Check the section C-API incompatibility at the Troubleshooting ImportError section at https://numpy.org/devdocs/user/troubleshooting-importerror.html#c-api-incompatibility for indications on how to solve this problem.

In [2]:
# Load the dataset
data_path = "dataset.csv"
data = pd.read_csv(data_path)

# Shuffle the dataset
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Extract features (X) and labels (y)
X = data.drop(columns=["label"])  # Exclude 'label' and 'time_stamp_0'
y = data["label"]

# Encode the labels
y = pd.factorize(y)[0]  # Convert categorical labels to numeric

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

In [3]:
# Train an XGBoost classifier
model = XGBClassifier(
    n_estimators=50,       # Number of trees
    max_depth=12,            # Maximum tree depth
    learning_rate=0.1,      # Learning rate for gradient boosting
    eval_metric="mlogloss", # Multi-class log loss
    random_state=42,
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=12,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=None, num_parallel_tree=None, objective='multi:softprob', ...)

In [4]:
# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Save the model for deployment
#model.save_model("sign_language_xgboost_model.json")
print("Model saved successfully!")

Test Accuracy: 1.0000
Model saved successfully!


<br><br><br>
## LightGBM 

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import lightgbm as lgb

# Load the dataset
data_path = "dataset.csv"
data = pd.read_csv(data_path)

data = data.loc[:, ~data.columns.str.contains("timestamp", case=False)]

# Shuffle the dataset
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Extract features (X) and labels (y)
X = data.drop(columns=["label"])  # Exclude 'label' and 'time_stamp_0'
y = data["label"]

# Encode the labels
y = pd.factorize(y)[0]  # Convert categorical labels to numeric

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Create a LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Define parameters for LightGBM
params = {
    "objective": "multiclass",
    "num_class": len(pd.unique(y)),  # Number of classes
    "boosting_type": "gbdt",
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 15,
    "max_depth": 5,
    "verbosity": -1,
    "random_state": 42,
     "min_data_in_leaf": 20
}

# Train the model
model = lgb.train(params, train_data, valid_sets=[test_data], num_boost_round=1200)

# Make predictions
y_pred = model.predict(X_test)
y_pred = [list(pred).index(max(pred)) for pred in y_pred]  # Convert probabilities to class indices

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Save the model for deployment
model.save_model("model_lightgbm.txt")
#print("Model saved successfully!")

Test Accuracy: 0.9808


<br><br><br>

## RandomForest

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib

# Load the dataset
data_path = "dataset.csv"
data = pd.read_csv(data_path)

# Shuffle the dataset
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Extract features (X) and labels (y)
X = data.drop(columns=["label"])  # Drop label and timestamp
y = data["label"]

# Encode the labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Train the Random Forest model
model = RandomForestClassifier(
    n_estimators=25,  # Number of trees
    max_depth=None,    # Let trees grow until all leaves are pure or contain less than min_samples_split
    random_state=42
)
model.fit(X_train, y_train)

# Make predictions on test data
y_test_pred = model.predict(X_test)

# Calculate and print test accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Save the model
#joblib.dump(model, "random_forest_sign_language_model.pkl")
#print("Model saved successfully!")

# Save the label encoder for decoding predictions later
#joblib.dump(label_encoder, "label_encoder.pkl")

Test Accuracy: 0.9923


## ANN

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Load the dataset
data_path = "dataset.csv"
data = pd.read_csv(data_path)

# Drop timestamp columns if they exist
data = data.loc[:, ~data.columns.str.contains("timestamp", case=False)]

# Shuffle the dataset
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Extract features (X) and labels (y)
X = data.drop(columns=["label"])
y = data["label"]

# Encode the labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Standardize the features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Get the number of unique classes
num_classes = len(np.unique(y))

# Convert labels to one-hot encoding
y_train_one_hot = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_one_hot = tf.keras.utils.to_categorical(y_test, num_classes)

# Build the ANN model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train_one_hot, validation_data=(X_test, y_test_one_hot), epochs=50, batch_size=32, verbose=1)

# Evaluate the model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

accuracy = accuracy_score(y_test, y_pred_classes)
print(f"Test Accuracy: {accuracy:.4f}")

# Save the model for deployment
model.save("ann_model.h5")
print("Model saved successfully!")

2025-01-02 15:02:04.163761: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-02 15:02:04.178169: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-02 15:02:04.297080: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-02 15:02:04.298360: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-02 15:02:04.855438: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not fin

Epoch 1/50


2025-01-02 15:02:05.542625: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-01-02 15:02:05.542959: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


13/13 [==============================] - 0s 9ms/step - loss: 1.3081 - accuracy: 0.4744 - val_loss: 0.4972 - val_accuracy: 0.9154
Epoch 2/50
13/13 [==============================] - 0s 2ms/step - loss: 0.4433 - accuracy: 0.8744 - val_loss: 0.1573 - val_accuracy: 0.9769
Epoch 3/50
13/13 [==============================] - 0s 2ms/step - loss: 0.2048 - accuracy: 0.9564 - val_loss: 0.0795 - val_accuracy: 0.9769
Epoch 4/50
13/13 [==============================] - 0s 2ms/step - loss: 0.1129 - accuracy: 0.9692 - val_loss: 0.0467 - val_accuracy: 0.9846
Epoch 5/50
13/13 [==============================] - 0s 2ms/step - loss: 0.0853 - accuracy: 0.9744 - val_loss: 0.0321 - val_accuracy: 0.9846
Epoch 6/50
13/13 [==============================] - 0s 2ms/step - loss: 0.0405 - accuracy: 0.9872 - val_loss: 0.0268 - val_accuracy: 0.9846
Epoch 7/50
13/13 [==============================] - 0s 2ms/step - loss: 0.0408 - accuracy: 0.9923 - val_loss: 0.0242 - val_accuracy: 0.9846
Epoch 8/50
13/13 [=============

/home/s4lm-xi/anaconda3/lib/python3.9/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
